# IMPORT

In [39]:
%load_ext autoreload
%autoreload 2

from importlib import reload
import my_sklearn
reload(my_sklearn)

from my_sklearn import my_MLP, py_MLP
import torch


import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from category_encoders import CountEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from itertools import product
from sklearn.neural_network import MLPClassifier
from joblib import Parallel, delayed

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# LOAD DATA AND PREPROCESSING

load data from [kaggle](https://www.kaggle.com/competitions/DontGetKicked/data?select=training.csv)

In [40]:
df = pd.read_csv("./data/training.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 72983 entries, 0 to 72982
Data columns (total 34 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   RefId                              72983 non-null  int64  
 1   IsBadBuy                           72983 non-null  int64  
 2   PurchDate                          72983 non-null  str    
 3   Auction                            72983 non-null  str    
 4   VehYear                            72983 non-null  int64  
 5   VehicleAge                         72983 non-null  int64  
 6   Make                               72983 non-null  str    
 7   Model                              72983 non-null  str    
 8   Trim                               70623 non-null  str    
 9   SubModel                           72975 non-null  str    
 10  Color                              72975 non-null  str    
 11  Transmission                       72974 non-null  str    
 12  W

In [41]:
df.head()

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,12/7/2009,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,...,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,12/7/2009,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,...,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053
2,3,0,12/7/2009,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,...,7146.0,8702.0,NaN,NaN,19638,33619,FL,4900.0,0,1389
3,4,0,12/7/2009,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,...,4375.0,5518.0,NaN,NaN,19638,33619,FL,4100.0,0,630
4,5,0,12/7/2009,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,...,6739.0,7911.0,NaN,NaN,19638,33619,FL,4000.0,0,1020


#### Design the train/validation/test split

In [42]:
df['PurchDate'] = pd.to_datetime(df["PurchDate"]).dt.date
dates = np.sort(df['PurchDate'].unique())
number_of_dates = len(dates)

valid_date = dates[number_of_dates // 3]
test_date = dates[2 * number_of_dates // 3]

train_mask = df["PurchDate"] < valid_date
valid_mask = (df["PurchDate"] >= valid_date) & (df['PurchDate'] < test_date)
test_mask = df['PurchDate'] >= test_date

train_df = df[train_mask]
valid_df = df[valid_mask]
test_df = df[test_mask]

X_train = train_df.drop(columns='IsBadBuy')
y_train = train_df['IsBadBuy'].astype('int8')

X_valid = valid_df.drop(columns='IsBadBuy')
y_valid = valid_df['IsBadBuy'].astype('int8')

X_test = test_df.drop(columns='IsBadBuy')
y_test = test_df['IsBadBuy'].astype('int8')


### Check

In [43]:
print(f"length of df = {len(df)}, sum of lengths of split = {len(X_train) + len(X_valid) + len(X_test)}")
print()
print(f"max date of TRAIN = {X_train['PurchDate'].max()}\nmin date of VALID = {X_valid['PurchDate'].min()}")
print()
print(f"max date of VALID = {X_valid['PurchDate'].max()}\nmin date of TEST = {X_test['PurchDate'].min()}")

length of df = 72983, sum of lengths of split = 72983

max date of TRAIN = 2009-09-01
min date of VALID = 2009-09-02

max date of VALID = 2010-04-30
min date of TEST = 2010-05-03


### Preprocess

In [44]:
drop_columns = ['RefId', 'PurchDate', 'VehYear', 'WheelTypeID', 'PRIMEUNIT', 'AUCGUART', 'VNZIP1', 'BYRNO', 'Auction']

X_train = X_train.drop(columns=drop_columns)
X_valid = X_valid.drop(columns=drop_columns)
X_test = X_test.drop(columns=drop_columns)

In [45]:
unique_number_cat = X_train.select_dtypes(include='str').nunique()
unique_number_cat

Make                     31
Model                   795
Trim                    122
SubModel                650
Color                    16
Transmission              2
WheelType                 3
Nationality               4
Size                     12
TopThreeAmericanName      4
VNST                     29
dtype: int64

In [46]:
X_train.select_dtypes(include=['int64']).nunique()

VehicleAge          8
VehOdo          18432
IsOnlineSale        1
WarrantyCost      238
dtype: int64

In [47]:
categorical_feat_count = [ind for ind in unique_number_cat.index.to_list() if unique_number_cat[ind] > 4]

categorical_feat_onehot = [ind for ind in unique_number_cat.index.to_list() if unique_number_cat[ind] <= 4]
categorical_feat_onehot.append('IsOnlineSale')

number_features = X_train.select_dtypes(include='number').columns.to_list()
number_features.remove("IsOnlineSale")

In [48]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat_count',
            Pipeline([
                ('encdoer', CountEncoder(handle_missing='unknown', handle_unknown=0)),
                ('min_max', MinMaxScaler())
            ]),
            categorical_feat_count),
        ('ohe',
            Pipeline([
                ('impute', SimpleImputer(strategy='constant', fill_value='unknown')),
                ('ohe_in', OneHotEncoder(handle_unknown='ignore'))
            ]),
            categorical_feat_onehot),
        ('num',
            Pipeline([
                ('impute', SimpleImputer(strategy='median')),
                ('min_max', MinMaxScaler())
            ]),
            number_features)
    ]
)

In [49]:
preprocessor.fit(X_train)
preprocessor.set_output()

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat_count', ...), ('ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fea

In [50]:
def optimize_dtypes(df):
    for col in df.columns:
        if np.array_equal(df[col], df[col].astype(int)):
            max_val = df[col].max()
            min_val = df[col].min()
            if min_val >= -128 and max_val <= 127:
                df[col] = df[col].astype('int8')
            else:
                df[col] = df[col].astype('int16')
        else:
            df[col] = df[col].astype('float32')
    return df

In [51]:
X_train_processed = pd.DataFrame(
    preprocessor.transform(X_train),
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index
)
X_train_processed = optimize_dtypes(X_train_processed)

X_valid_processed = pd.DataFrame(
    preprocessor.transform(X_valid),
    columns=preprocessor.get_feature_names_out(),
    index=X_valid.index
)
X_valid_processed = optimize_dtypes(X_valid_processed)

X_test_processed = pd.DataFrame(
    preprocessor.transform(X_test),
    columns=preprocessor.get_feature_names_out(),
    index=X_test.index
)
X_test_processed = optimize_dtypes(X_test_processed)

# MLP

In [52]:
def evaluate(model_name, model, params, X_train, y_train, X_valid, y_valid):

    model = model(**params)
    model.fit(X_train, y_train)

    y_proba_train = model.predict_proba(X_train)[:, 1]
    y_proba_valid = model.predict_proba(X_valid)[:, 1]

    gini_train = 2 * roc_auc_score(y_train, y_proba_train) - 1
    gini_valid = 2 * roc_auc_score(y_valid, y_proba_valid) - 1

    return {
        'model_name': model_name,
        'params': params,
        'Gini_train': gini_train,
        'Gini_valid': gini_valid,
        'Gini_gap': gini_train - gini_valid
    }

def grid_search(model_name, model, param_grid, X_train, y_train, X_valid, y_valid, n_jobs=-1):

    param_names = list(param_grid.keys())
    param_values = list(param_grid.values())

    param_combinations = [
        dict(zip(param_names, values))
        for values in product(*param_values)
    ]
    
    results = Parallel(n_jobs=n_jobs)(
        delayed(evaluate)(
            model_name,
            model,
            params,
            X_train,
            y_train,
            X_valid,
            y_valid
        )
        for params in param_combinations
    )

    return pd.DataFrame(results)

### My MLP

In [53]:
param_grid = {
    'n_hidden': [100, 500, 1000, 2000],
    'activation': ['relu', 'sigmoid', 'cosine', 'tanh'],
    'learning_rate': [0.001, 0.01],
    'max_iter': [3000],
    'solver': ['adam', 'sgd'],
    'early_stopping': [True],
    'validation_fraction' : [0.1, 0.15, 0.2],
    'random_state': [21]
}

results_my = grid_search(
    'my_MLP',
    my_MLP,
    param_grid,
    X_train_processed,
    y_train,
    X_valid_processed,
    y_valid
)

results_my = results_my.sort_values(
    'Gini_valid',
    ascending=False
)

results_my


,model_name,params,Gini_train,Gini_valid,Gini_gap
104,my_MLP,"{'n_hidden': 1000, 'activation': 'relu', 'lear...",0.505901,0.475904,0.029998
20,my_MLP,"{'n_hidden': 100, 'activation': 'sigmoid', 'le...",0.492854,0.475156,0.017698
122,my_MLP,"{'n_hidden': 1000, 'activation': 'cosine', 'le...",0.493260,0.473219,0.020040
168,my_MLP,"{'n_hidden': 2000, 'activation': 'cosine', 'le...",0.492668,0.472655,0.020012
116,my_MLP,"{'n_hidden': 1000, 'activation': 'sigmoid', 'l...",0.499353,0.472372,0.026981
...,...,...,...,...,...
28,my_MLP,"{'n_hidden': 100, 'activation': 'cosine', 'lea...",-0.056502,-0.087498,0.030997
129,my_MLP,"{'n_hidden': 1000, 'activation': 'cosine', 'le...",-0.112716,-0.149617,0.036901
123,my_MLP,"{'n_hidden': 1000, 'activation': 'cosine', 'le...",-0.122497,-0.160232,0.037735
125,my_MLP,"{'n_hidden': 1000, 'activation': 'cosine', 'le...",-0.130364,-0.168067,0.037704


### SKLEARN MLP WITH 1 LAYER

In [54]:
param_grid_sklearn = {
    'hidden_layer_sizes': [(100,), (500,), (1000,), (2000,)],
    'activation': ['relu', 'logistic', 'tanh'],
    'learning_rate_init': [0.001, 0.01],
    'max_iter': [3000],
    'solver': ['adam', 'sgd'],
    'validation_fraction': [0.1, 0.15, 0.2],
    'early_stopping': [True],
    'random_state': [21]
}

results_sklearn = grid_search(
    'sklearn_MLP',
    MLPClassifier,
    param_grid_sklearn,
    X_train_processed,
    y_train,
    X_valid_processed,
    y_valid
)

results_sklearn = results_sklearn.sort_values(
    'Gini_valid',
    ascending=False
)

results_sklearn

,model_name,params,Gini_train,Gini_valid,Gini_gap
20,sklearn_MLP,"{'hidden_layer_sizes': (100,), 'activation': '...",0.496413,0.474046,0.022367
30,sklearn_MLP,"{'hidden_layer_sizes': (100,), 'activation': '...",0.496005,0.473612,0.022393
6,sklearn_MLP,"{'hidden_layer_sizes': (100,), 'activation': '...",0.498376,0.473055,0.025320
44,sklearn_MLP,"{'hidden_layer_sizes': (500,), 'activation': '...",0.508620,0.472866,0.035754
32,sklearn_MLP,"{'hidden_layer_sizes': (100,), 'activation': '...",0.504889,0.471675,0.033214
...,...,...,...,...,...
4,sklearn_MLP,"{'hidden_layer_sizes': (100,), 'activation': '...",0.084734,0.072590,0.012144
5,sklearn_MLP,"{'hidden_layer_sizes': (100,), 'activation': '...",0.077118,0.066665,0.010453
51,sklearn_MLP,"{'hidden_layer_sizes': (500,), 'activation': '...",0.046715,0.030139,0.016576
52,sklearn_MLP,"{'hidden_layer_sizes': (500,), 'activation': '...",0.044721,0.028133,0.016587


### SKLEARN MLP WITH SEVERAL LAYERS

In [55]:
param_grid_sklearn_multi = {
    'hidden_layer_sizes': [(100, 100), (500, 200), (100, 200, 100), (20, 20, 20, 20)],
    'activation': ['relu', 'logistic', 'tanh'],
    'learning_rate_init': [0.001, 0.01],
    'max_iter': [3000],
    'solver': ['adam', 'sgd'],
    'validation_fraction': [0.1, 0.15, 0.2],
    'early_stopping': [True],
    'random_state': [21]
}

results_sklearn_multi = grid_search(
    'sklearn_MLP_multi',
    MLPClassifier,
    param_grid_sklearn_multi,
    X_train_processed,
    y_train,
    X_valid_processed,
    y_valid
)

results_sklearn_multi = results_sklearn_multi.sort_values(
    'Gini_valid',
    ascending=False
)

results_sklearn_multi

,model_name,params,Gini_train,Gini_valid,Gini_gap
80,sklearn_MLP_multi,"{'hidden_layer_sizes': (100, 200, 100), 'activ...",0.509965,0.476653,0.033312
119,sklearn_MLP_multi,"{'hidden_layer_sizes': (20, 20, 20, 20), 'acti...",0.497512,0.475278,0.022234
56,sklearn_MLP_multi,"{'hidden_layer_sizes': (500, 200), 'activation...",0.499793,0.475263,0.024530
79,sklearn_MLP_multi,"{'hidden_layer_sizes': (100, 200, 100), 'activ...",0.509338,0.474803,0.034534
20,sklearn_MLP_multi,"{'hidden_layer_sizes': (100, 100), 'activation...",0.494686,0.473787,0.020899
...,...,...,...,...,...
94,sklearn_MLP_multi,"{'hidden_layer_sizes': (100, 200, 100), 'activ...",-0.123060,-0.127899,0.004839
95,sklearn_MLP_multi,"{'hidden_layer_sizes': (100, 200, 100), 'activ...",-0.123384,-0.128634,0.005250
87,sklearn_MLP_multi,"{'hidden_layer_sizes': (100, 200, 100), 'activ...",-0.149715,-0.153891,0.004175
89,sklearn_MLP_multi,"{'hidden_layer_sizes': (100, 200, 100), 'activ...",-0.149882,-0.154050,0.004168


### Pytorch MLP

In [56]:
X_train_torch = torch.tensor(X_train_processed.values, dtype=torch.float32)
y_train_torch = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)

X_valid_torch = torch.tensor(X_valid_processed.values, dtype=torch.float32)
y_valid_torch = torch.tensor(y_valid.values, dtype=torch.float32).reshape(-1, 1)

In [57]:
param_grid_pytorch = {
    'n_features': [X_train_torch.shape[1]],
    'hidden_layers': [
        [100],
        [500],
        [500, 100],
        [500, 200],
        [100, 100, 100]
    ],
    'activation': ['relu', 'tanh'],
    'learning_rate': [0.001, 0.01],
    'optimizer': ['adam', 'sgd'],
}

results_pytorch = grid_search(
    'pytorch_MLP',
    py_MLP,
    param_grid_pytorch,
    X_train_torch,
    y_train_torch,
    X_valid_torch,
    y_valid_torch
)

results_pytorch = results_pytorch.sort_values(
    'Gini_valid',
    ascending=False
)

results_pytorch

/home/apkin/school21/projects/ML7_Clustering_algorithms_ID_1254806-1/src/venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


,model_name,params,Gini_train,Gini_valid,Gini_gap
12,pytorch_MLP,"{'n_features': 35, 'hidden_layers': [500], 'ac...",0.506876,0.477445,0.029430
4,pytorch_MLP,"{'n_features': 35, 'hidden_layers': [100], 'ac...",0.506493,0.476212,0.030281
2,pytorch_MLP,"{'n_features': 35, 'hidden_layers': [100], 'ac...",0.507819,0.473747,0.034072
39,pytorch_MLP,"{'n_features': 35, 'hidden_layers': [100, 100,...",0.494890,0.470557,0.024333
35,pytorch_MLP,"{'n_features': 35, 'hidden_layers': [100, 100,...",0.501193,0.469797,0.031395
20,pytorch_MLP,"{'n_features': 35, 'hidden_layers': [500, 100]...",0.524814,0.469270,0.055544
10,pytorch_MLP,"{'n_features': 35, 'hidden_layers': [500], 'ac...",0.509836,0.469153,0.040683
37,pytorch_MLP,"{'n_features': 35, 'hidden_layers': [100, 100,...",0.483899,0.468898,0.015001
6,pytorch_MLP,"{'n_features': 35, 'hidden_layers': [100], 'ac...",0.506347,0.468445,0.037901
23,pytorch_MLP,"{'n_features': 35, 'hidden_layers': [500, 100]...",0.494996,0.468380,0.026616


# Take the best model and estimate its performance on the test dataset.

In [67]:
best_params = results_pytorch.iloc[0]['params']

model = py_MLP(**best_params)

In [69]:
best_params

{'n_features': 35,
 'hidden_layers': [500],
 'activation': 'tanh',
 'learning_rate': 0.001,
 'optimizer': 'adam'}

In [68]:
X_test_torch = torch.tensor(X_test_processed.values, dtype=torch.float32)
y_test_torch = torch.tensor(y_test.values, dtype=torch.float32)

model.fit(X_train_torch, y_train_torch)

y_proba_train = model.predict_proba(X_train_torch)[:, 1]
y_proba_valid = model.predict_proba(X_valid_torch)[:, 1]
y_proba_test = model.predict_proba(X_test_torch)[:, 1]

gini_train = 2 * roc_auc_score(y_train, y_proba_train) - 1
gini_valid = 2 * roc_auc_score(y_valid, y_proba_valid) - 1


gini_test = 2 * roc_auc_score(y_test_torch, y_proba_test) - 1

print(f"Gini train: {gini_train}")
print(f"Gini valid: {gini_valid}")
print(f"Gini test: {gini_test}")

Gini train: 0.5004002185813505
Gini valid: 0.4701935979562484
Gini test: 0.4789081020583703


All three approaches — my own MLP implementation, sklearn MLPClassifier, and PyTorch MLP — showed similar results on the validation dataset. The PyTorch implementation achieved the best validation performance among them.

For the final PyTorch model, the Gini scores were:

* Train: 0.5004
* Validation: 0.4702
* Test: 0.4789

The difference between the validation and test Gini scores is very small, and the test score is even slightly higher than the validation score. This suggests that the model generalizes well to unseen data and does not show significant overfitting. The results are also quite consistent across the three datasets.
